# Heidelberg

In [5]:
import pandas as pd 
import matplotlib.pyplot as plt
from ipynb.fs.full.get_table import get_table

# get hourly weather data for Heidelberg (2024-11-01 to 2025-10-31)
hd_weather_hourly = pd.read_csv('weather_data/hourly/heidelberg_weather_2024-11-01_2025-10-31.csv')

# get hourly bike count data for Heidelberg (2024-11-01 to 2025-10-31)
hd_bike_hourly =  get_table('eco-counter/all_cities', 2024, 11, 1, 2025, 10, 31)
hd_bike_hourly = hd_bike_hourly[hd_bike_hourly['counter_site'] == "Mannheimer Straße"]

# Add a new column datetime in both tables (for same name)
hd_weather_hourly['datetime'] = pd.to_datetime(hd_weather_hourly['time'])
hd_bike_hourly['datetime']    = pd.to_datetime(hd_bike_hourly['iso_timestamp'])

# Get the common datetimes
common_ts = hd_weather_hourly['datetime'][hd_weather_hourly['datetime'].isin(hd_bike_hourly['datetime'])]

# Make a new table with all necessary data that is in both sets
df_common = pd.DataFrame({
    'bike': hd_bike_hourly[hd_bike_hourly['datetime'].isin(common_ts)]['channels_all'].values,
    'rain': hd_weather_hourly[hd_weather_hourly['datetime'].isin(common_ts)]['prcp'].values,
    'temp': hd_weather_hourly[hd_weather_hourly['datetime'].isin(common_ts)]['temp'].values
})

# Drop any missing numbers like NaNs
df_common = df_common.apply(pd.to_numeric, errors='coerce').dropna()

print(f'Hourly weather data points: {len(hd_weather_hourly)}')
print(f'Hourly bike data points for Heidelberg: {len(hd_bike_hourly)}')
print("Final data size:", len(df_common))
print(df_common.corr())

# Calculate the missing dates
missing_in_bike = hd_weather_hourly.loc[~hd_weather_hourly['datetime'].isin(hd_bike_hourly['datetime']), 'datetime']
missing_in_weather = hd_bike_hourly.loc[~hd_bike_hourly['datetime'].isin(hd_weather_hourly['datetime']), 'datetime']

print("Missing bike data, only date:", len(missing_in_bike))
print("Missing weather data, only date:", len(missing_in_weather))

print("Examples of missing data:", missing_in_bike[:10])


Hourly weather data points: 8760
Hourly bike data points for Heidelberg: 8709
Final data size: 8464
          bike      rain      temp
bike  1.000000 -0.060540  0.303624
rain -0.060540  1.000000  0.014875
temp  0.303624  0.014875  1.000000
Missing bike data, only date: 52
Missing weather data, only date: 1
Examples of missing data: 719    2024-11-30 23:00:00
1463   2024-12-31 23:00:00
2207   2025-01-31 23:00:00
2879   2025-02-28 23:00:00
3621   2025-03-31 22:00:00
4341   2025-04-30 22:00:00
5085   2025-05-31 22:00:00
5805   2025-06-30 22:00:00
6549   2025-07-31 22:00:00
7274   2025-08-31 03:00:00
Name: datetime, dtype: datetime64[ns]
